# RME ASR v3 — benchmark berpasangan pada Colab T4
Jalankan setelah notebook training v3. Membandingkan base Whisper dan model baru pada **klip identik**,
konfigurasi decoding identik, dengan CI bootstrap per kelompok independen yang diketahui.
Kode CPU telah diuji; inference GPU dan hasil model baru belum diukur oleh penyusun.

Fokus: akurasi transkripsi untuk draft RME. LLM/SOAP dari notebook lama tidak dijadikan ground truth;
kesamaan SOAP dari dua transkrip bukan bukti kebenaran klinis. Model v2 boleh ditambahkan sebagai diagnostik,
tetapi split v3 tidak otomatis merupakan holdout untuk v2 yang sudah dilatih sebelumnya.

## 1. Dependensi

In [ ]:
import subprocess, sys
PACKAGES = ['transformers==4.57.3','peft==0.18.0','accelerate==1.12.0',
            'numpy==2.2.6','scipy==1.15.3','soundfile==0.13.1','librosa==0.11.0',
            'gdown==5.2.0','pytest==8.4.2']
subprocess.run([sys.executable,'-m','pip','install','-q',*PACKAGES],check=True)
# Jika Colab meminta restart, restart runtime lalu Run all. Torch/CUDA bawaan tidak diganti.

## 2. Konfigurasi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, pathlib, torch
from pathlib import Path
os.chdir('/content')
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab dahulu')
ROOT = Path('/content/gold_rme_v3')
WORK = Path('/content/drive/MyDrive/rme_asr_v3')
WORK.mkdir(parents=True,exist_ok=True)
ARCHIVE = Path('/content/gold_dataset.tar.gz')
FILE_ID = '1B551Fgb98EaK0Bx-g4BaCogzB3QjOjfR'
EXPECTED_ARCHIVE_SHA256 = '1213c8df5ca041dbf7dcfb119cd2d9112d970ad0849cff2193f02d1dd65c68d6'
SEED = 13
print(torch.cuda.get_device_name(0), 'torch',torch.__version__,'CUDA',torch.version.cuda)

## 3. Modul bersama dan tes

In [ ]:
%%writefile rme_core.py
"""RME ASR v3: shared data contracts and transparent metrics. No clinical auto-correction."""
import collections, hashlib, json, math, re, tarfile, unicodedata
from pathlib import Path
import numpy as np

SR = 16000
BASE = 'openai/whisper-large-v3-turbo'

def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def write_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix(p.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    tmp.replace(p)

def normalize(text):
    # Keep decimals, dose separators, percent and negation. No synonym/filler rewriting.
    t = unicodedata.normalize('NFKC', str(text)).lower()
    t = re.sub(r'(?<=\d),(?=\d)', '.', t)
    t = re.sub(r'(?<!\d)\.|\.(?!\d)', ' ', t)
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s./%+\-]', ' ', t)).strip()

def extract_safe(archive, dest):
    dest = Path(dest).resolve(); dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive) as t:
        for m in t:
            p = (dest/m.name).resolve()
            if not p.is_relative_to(dest) or not (m.isfile() or m.isdir()):
                raise ValueError('Unsafe archive member: ' + m.name)
            t.extract(m, dest, filter='data')

def read_records(path):
    p = Path(path)
    if p.suffix == '.jsonl': return [json.loads(x) for x in p.read_text().splitlines() if x.strip()]
    if p.suffix == '.csv':
        import csv
        return list(csv.DictReader(p.open()))
    obj = json.loads(p.read_text())
    if isinstance(obj, list): return obj
    for key in ('data', 'records', 'samples', 'rows'):
        if isinstance(obj.get(key), list): return obj[key]
    raise ValueError(f'Unsupported manifest structure: {p}')

def canonical_rows(manifests, root):
    root = Path(root).resolve()
    files = collections.defaultdict(list)
    for p in root.rglob('*'):
        if p.suffix.lower() in ('.wav','.mp3','.flac','.ogg'): files[p.name].append(p)
    rows = []; seen = set()
    for manifest in manifests:
        for x in read_records(manifest):
            name = x.get('audio_file') or x.get('audio_path') or x.get('path') or x.get('file')
            text = x.get('raw_text', x.get('text', x.get('transcript', x.get('ref'))))
            if isinstance(text,str): text=re.sub(r'</?[A-Za-z][^>]*>', '', text)  # remove corpus language tags only
            if not name or not isinstance(text,str): raise ValueError('Manifest needs audio_file and text')
            name = str(name); candidates = [root/name, Path(manifest).parent/name]
            found = [p.resolve() for p in candidates if p.is_file() and p.resolve().is_relative_to(root)]
            if not found:
                found = files.get(Path(name).name, [])
                if len(found) != 1: raise ValueError('Missing/ambiguous audio: ' + name)
            p = found[0]; rel = str(p.relative_to(root))
            if rel in seen: raise ValueError('Duplicate manifest row: ' + rel)
            seen.add(rel)
            stem = p.stem
            syn = stem.startswith('SYN_') or str(x.get('source','')).lower() in ('synthetic','tts')
            source = 'synthetic' if syn else 'medisco' if stem.startswith('medisco_') else 'video' if re.match(r'(lvl3|clean)_chk_',stem) else 'fleurs' if stem.startswith('fleurs_') else str(x.get('source','unknown'))
            video = str(x.get('video_id') or '')
            if not video:
                match = re.match(r'(?:lvl3|clean)_chk_\d+_(.{11})$', stem)
                if match: video = match[1]  # YouTube IDs may contain underscores.
            speaker = str(x.get('speaker_id') or '')
            if source == 'medisco' and not speaker:
                match = re.search(r'(?:female|male)(?:-\d+)?',stem)
                if match: speaker = 'medisco:' + match[0]
            # Synthetic template family is training only; never use it as a human speaker ID.
            group = str(x.get('session_id') or video or speaker or x.get('group_id') or '')
            if syn: group = 'synthetic'
            if source=='fleurs':group='fleurs_unknown_speakers'  # per-clip IDs do not establish speaker independence
            if not group: raise ValueError('Missing recording/speaker/session group: ' + rel)
            declared = next((v for v in Path(rel).parts if v in ('train','dev','test')), 'unknown')
            clean=x.get('clean_text',text)
            changes=critical_proxy(text,clean)
            review_flags=[f'clean_changed_{k}' for k,v in changes.items() if v]
            if normalize(text)!=normalize(clean):review_flags.append('clean_lexical_change')
            rows.append(dict(id=rel, audio_file=rel, ref=text.strip(), source=source, synthetic=syn,
                training_only=syn or source=='fleurs',source_detail=x.get('source','unknown'),
                clean_text=clean,label_review_flags=review_flags,reference_policy='source_raw_without_language_tags',
                video_id=video, speaker_id=speaker, session_id=str(x.get('session_id') or ''),
                patient_id=str(x.get('patient_id') or ''), group_id=group, declared_split=declared,
                category=x.get('clinical_category',x.get('topic','unknown')),
                declared_seconds=float(x.get('duration',x.get('duration_seconds',0)) or 0),
                reference_verified=x.get('reference_verified') is True))
    if not rows: raise ValueError('Empty dataset')
    return sorted(rows,key=lambda r:r['id'])

def inspect_audio(row, root):
    import soundfile as sf
    r = dict(row); issues = []
    if any(f in row.get('label_review_flags',[]) for f in ('clean_changed_number','clean_changed_unit','clean_changed_negation')):
        issues.append('label_critical_disagreement_requires_audio_review')
    try:
        path = Path(root)/r['audio_file']
        x, sr = sf.read(path, dtype='float32', always_2d=True)
        r.update(sr=sr, channels=x.shape[1], seconds=len(x)/sr,
                 audio_sha256=file_hash(path), pcm_sha256=hashlib.sha256(x.tobytes()).hexdigest())
        r['peak'] = float(np.max(np.abs(x))) if len(x) else 0.
        r['rms'] = float(np.sqrt(np.mean(x*x))) if len(x) else 0.
        r['clip_fraction'] = float(np.mean(np.abs(x)>=.999)) if len(x) else 0.
        if not np.isfinite(x).all(): issues.append('nonfinite')
        if not 0.2 <= r['seconds'] <= 30.: issues.append('duration_out_of_window')
        if r.get('declared_seconds',0)>0 and abs(r['seconds']-r['declared_seconds'])>max(.5,.1*r['declared_seconds']):
            issues.append('duration_metadata_mismatch_requires_review')
        if r['rms'] < 1e-5 and r['ref']: issues.append('silent_with_label')
        if not r['ref']: issues.append('empty_label_requires_nonspeech_verification')
        r['warnings'] = []
        wps = len(normalize(r['ref']).split())/max(r['seconds'],.001)
        if wps < .6 or wps > 4.5: r['warnings'].append('unusual_label_rate')
        if r['clip_fraction'] > .001: r['warnings'].append('clipping')
        words = normalize(r['ref']).split()
        if len(words)>10 and len(set(words))/len(words)<.25: r['warnings'].append('repetition')
    except Exception as e: issues.append('decode_error:' + type(e).__name__)
    r['reject_reasons'] = issues
    return r

def grouped_split(rows, seed=13):
    """Connected components prevent shared recordings/speakers/patients/audio in different splits."""
    parent = list(range(len(rows)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    seen = {}
    for i,r in enumerate(rows):
        for key in ('group_id','video_id','speaker_id','session_id','patient_id','pcm_sha256'):
            val=r.get(key)
            if not val: continue
            k=(key,val)
            if k in seen: parent[find(i)] = find(seen[k])
            seen[k]=i
    members=collections.defaultdict(list)
    for i,r in enumerate(rows): members[find(i)].append(r)
    out=[]
    for group in members.values():
        identities=sorted({(key,str(r[key])) for r in group for key in ('group_id','video_id','speaker_id','session_id','patient_id') if r.get(key)})
        gid=digest(identities)  # stable when clips within the same known source are added/removed
        u=int(digest([seed,gid])[:12],16)/16**12
        split='train' if any(r.get('training_only',r['synthetic']) for r in group) else 'test' if u<.1 else 'dev' if u<.2 else 'train'
        out.extend(dict(r,split=split,cluster_id=gid) for r in group)
    # Ensure each sufficiently large non-synthetic source has at least one group per split.
    # This is metadata-only and deterministic; never use model scores to choose a split.
    strata=collections.defaultdict(dict)
    for r in out:
        if not r.get('training_only',r['synthetic']):strata[r['source'] if 'source' in r else 'unknown'][r['cluster_id']]=r['split']
    for source,assign in strata.items():
        if len(assign)<3:continue
        for needed in ('train','dev','test'):
            counts=collections.Counter(assign.values())
            if counts[needed]:continue
            candidates=sorted(k for k,v in assign.items() if counts[v]>1)
            chosen=candidates[0];assign[chosen]=needed
        for r in out:
            if r['cluster_id'] in assign:r['split']=assign[r['cluster_id']]
    # Exact duplicates kept once per partition, train never learns held-out long references.
    refs={s:{normalize(r['ref']) for r in out if r['split']==s and len(normalize(r['ref']).split())>=8} for s in ('dev','test')}
    kept=[]; dropped=[]; pcm_seen=set()
    for r in sorted(out,key=lambda r:({'test':0,'dev':1,'train':2}[r['split']],r['id'])):
        ref=normalize(r['ref']); reason=None
        if r['split']=='train' and ref in refs['dev']|refs['test']: reason='heldout_exact_text'
        if r['split']=='dev' and ref in refs['test']: reason='test_exact_text'
        pcm=(r.get('sr'),r.get('pcm_sha256'))
        if pcm in pcm_seen: reason='duplicate_pcm'
        pcm_seen.add(pcm)
        (dropped if reason else kept).append(dict(r,drop_reason=reason) if reason else r)
    for s in ('train','dev','test'):
        if not any(r['split']==s for r in kept): raise ValueError('Empty split; insufficient independent groups')
    return kept,dropped

def edit_counts(a,b):
    # deterministic S/D/I counts, O(len(a)*len(b)) time and O(len(b)) memory
    prev=[(j,0,0,j) for j in range(len(b)+1)]
    for i,x in enumerate(a,1):
        cur=[(i,0,i,0)]
        for j,y in enumerate(b,1):
            c,s,d,k=prev[j-1]; sub=(c+(x!=y),s+(x!=y),d,k)
            c,s,d,k=prev[j]; delete=(c+1,s,d+1,k)
            c,s,d,k=cur[-1]; insert=(c+1,s,d,k+1)
            cur.append(min((sub,delete,insert), key=lambda z:z[0]))
        prev=cur
    _,s,d,i=prev[-1]; return int(s),int(d),int(i)

def score(ref,hyp):
    a,b=normalize(ref),normalize(hyp)
    s,d,i=edit_counts(a.split(),b.split()); cs,cd,ci=edit_counts(list(a),list(b))
    return dict(s=s,d=d,i=i,n=len(a.split()),err=s+d+i,chars=len(a),char_err=cs+cd+ci)

def paired_cluster_ci(rows, a, b, B=2000, seed=13):
    if set(a)!=set(b) or set(a)!={r['id'] for r in rows}: raise ValueError('Prediction coverage mismatch')
    clusters=collections.defaultdict(lambda:np.zeros(3))
    for r in rows:
        sa,sb=score(r['ref'],a[r['id']]),score(r['ref'],b[r['id']])
        clusters[r['cluster_id']]+=np.array([sa['err'],sb['err'],sa['n']])
    v=np.array(list(clusters.values())); total=v.sum(0)
    if not total[2]: raise ValueError('No reference words')
    result={'delta_wer_pp':float(100*(total[0]-total[1])/total[2]),'clusters':len(v),'unit':'connected_source_group','ci95':None}
    if len(v)<2: return result
    rng=np.random.default_rng(seed); boot=[]
    for _ in range(B):
        t=v[rng.integers(len(v),size=len(v))].sum(0)
        if t[2]>0: boot.append(100*(t[0]-t[1])/t[2])
    result['ci95']=np.quantile(boot,[.025,.975]).tolist();return result

def corpus_metrics(rows, hyps):
    if set(hyps)!={r['id'] for r in rows} or any(not isinstance(h,str) for h in hyps.values()):
        raise ValueError('All planned clips must have predictions; failures cannot disappear')
    totals=collections.Counter()
    for r in rows: totals.update(score(r['ref'],hyps[r['id']]))
    return dict(totals,wer=100*totals['err']/totals['n'] if totals['n'] else None,
                cer=100*totals['char_err']/totals['chars'] if totals['chars'] else None,clips=len(rows))

def critical_proxy(ref,hyp):
    # Screening only: no semantic scope/association inference or correctness certification.
    patterns={'number':r'\b\d+(?:[.,]\d+)?(?:/\d+)?\b',
              'unit':r'\b(?:mg|ml|mcg|kg|mmhg|gram|miligram|mililiter|persen|unit)\b',
              'negation':r'\b(?:tidak|tanpa|bukan|disangkal|negatif|belum)\b'}
    return {k:collections.Counter(re.findall(p,normalize(ref))) != collections.Counter(re.findall(p,normalize(hyp))) for k,p in patterns.items()}

def release_status(evidence):
    required=('independent_real_clinical_test','human_gold_references','critical_error_review',
              'silent_noise_test','long_form_test','latency_sla','subgroup_review','data_rights_review')
    missing=[k for k in required if evidence.get(k) is not True]
    return {'status':'NOT_READY' if missing else 'CANDIDATE_FOR_CONTROLLED_PILOT', 'missing':missing,
            'note':'Not automatic enterprise certification; clinician signs each RME draft.'}

def term_metrics(rows,hyps,terms):
    """Exact lexical occurrence scoring, not entity linking or semantic correctness."""
    terms=sorted({normalize(t) for t in terms if normalize(t)})
    if not terms:return {'status':'NOT_MEASURED','reason':'No clinician-approved lexicon supplied'}
    tp=fp=fn=0
    for r in rows:
        a,b=normalize(r['ref']),normalize(hyps[r['id']])
        for t in terms:
            p=r'(?<!\w)'+re.escape(t)+r'(?!\w)'
            na,nb=len(re.findall(p,a)),len(re.findall(p,b))
            tp+=min(na,nb);fp+=max(0,nb-na);fn+=max(0,na-nb)
    return {'tp':tp,'fp':fp,'fn':fn,'precision':tp/(tp+fp) if tp+fp else None,
            'recall':tp/(tp+fn) if tp+fn else None,'lexicon_size':len(terms),
            'note':'Lexical matches only; overlapping aliases must be curated before use.'}


def quarantine_conflicts(rows):
    groups=collections.defaultdict(list)
    for r in rows:
        if r.get('pcm_sha256'):groups[(r.get('sr'),r['pcm_sha256'])].append(r)
    for group in groups.values():
        if len({normalize(r['ref']) for r in group})>1:
            for r in group:
                reason='identical_audio_conflicting_references'
                if reason not in r['reject_reasons']:r['reject_reasons'].append(reason)
    return rows

In [ ]:
%%writefile rme_train.py
"""GPU training helpers; imported only after Colab dependencies are installed."""
import math, random
from dataclasses import dataclass
from pathlib import Path
import numpy as np
from rme_core import SR, BASE, corpus_metrics

def load_wave(path):
    import soundfile as sf
    from scipy.signal import resample_poly
    x,sr=sf.read(path,dtype='float32',always_2d=True)
    if not np.isfinite(x).all() or not 0 < len(x)/sr <= 30.: raise ValueError('Invalid audio: '+str(path))
    x=x.mean(axis=1)
    if sr!=SR:
        g=math.gcd(sr,SR);x=resample_poly(x,SR//g,sr//g)
    return np.asarray(x,dtype=np.float32)

class Clips:
    def __init__(self,rows,root,augment=False): self.rows,self.root,self.augment=rows,Path(root),augment
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        return {'path':str(self.root/r['audio_file']),'text':r['ref'],'augment':self.augment}

@dataclass
class Collator:
    processor: object
    rng: object = None
    _worker_seed: object = None
    def __call__(self,items):
        import torch
        worker_seed=torch.initial_seed() % 2**32
        if self.rng is None or self._worker_seed!=worker_seed:
            self.rng=np.random.default_rng(worker_seed);self._worker_seed=worker_seed
        waves=[]; ids=[]
        for it in items:
            x=load_wave(it['path'])
            if it['augment']:
                if self.rng.random()<.3:
                    rms=np.sqrt(np.mean(x*x))
                    x=x+self.rng.normal(0,rms*10**(-self.rng.uniform(20,35)/20),len(x)).astype(np.float32)
                if self.rng.random()<.3: x=x*10**(self.rng.uniform(-3,3)/20)
                peak=np.max(np.abs(x))
                if peak>.99:x=x*(.99/peak)
            waves.append(x)
            seq=self.processor.tokenizer(it['text']).input_ids
            if seq[0]==self.processor.tokenizer.convert_tokens_to_ids('<|startoftranscript|>'): seq=seq[1:]
            if len(seq)>448: raise ValueError('Label >448 tokens: segment and realign; never truncate')
            ids.append({'input_ids':seq})
        batch=self.processor.feature_extractor(waves,sampling_rate=SR,return_tensors='pt',return_attention_mask=True)
        labels=self.processor.tokenizer.pad(ids,return_tensors='pt')
        batch['labels']=labels['input_ids'].masked_fill(labels.attention_mask.ne(1),-100)
        return dict(batch)

def build_model(revision,r=16):
    import torch
    from transformers import WhisperForConditionalGeneration
    from peft import LoraConfig,get_peft_model
    model=WhisperForConditionalGeneration.from_pretrained(BASE,revision=revision,torch_dtype=torch.float32,attn_implementation='sdpa')
    model.config.use_cache=False
    model.config.apply_spec_augment=False  # mild waveform augmentation only; ablate on dev
    model.generation_config.language='indonesian';model.generation_config.task='transcribe'
    model.generation_config.forced_decoder_ids=None
    model.generation_config.max_length=448
    model.generation_config.use_cache=True
    return get_peft_model(model,LoraConfig(r=r,lora_alpha=2*r,lora_dropout=.05,
        target_modules=['q_proj','k_proj','v_proj','out_proj'],bias='none'))

def make_trainer(model,proc,train,dev,root,out,config):
    import torch
    from transformers import Seq2SeqTrainer,Seq2SeqTrainingArguments,EarlyStoppingCallback
    def metrics(pred):
        p=pred.predictions[0] if isinstance(pred.predictions,tuple) else pred.predictions
        p=np.where(p<0,proc.tokenizer.pad_token_id,p)
        labels=np.where(pred.label_ids<0,proc.tokenizer.pad_token_id,pred.label_ids)
        refs=proc.batch_decode(labels,skip_special_tokens=True)
        hyps=proc.batch_decode(p,skip_special_tokens=True)
        m=corpus_metrics([{'id':str(i),'ref':t} for i,t in enumerate(refs)],{str(i):t for i,t in enumerate(hyps)})
        return {'wer':m['wer'],'cer':m['cer']}
    steps=math.ceil(len(train)/(config['batch']*config['accum']))
    every=max(1,min(250,steps))
    args=Seq2SeqTrainingArguments(output_dir=str(out),
        per_device_train_batch_size=config['batch'],per_device_eval_batch_size=max(1,config['batch']//2),
        gradient_accumulation_steps=config['accum'],num_train_epochs=config['epochs'],
        learning_rate=config['lr'],lr_scheduler_type='cosine',warmup_ratio=.05,weight_decay=.01,
        max_grad_norm=1.,bf16=config['bf16'],fp16=not config['bf16'],
        gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},
        eval_strategy='steps',save_strategy='steps',eval_steps=every,save_steps=every,
        save_total_limit=2,load_best_model_at_end=True,metric_for_best_model='wer',greater_is_better=False,
        predict_with_generate=True,generation_max_length=448,generation_num_beams=1,
        eval_accumulation_steps=8,logging_steps=25,report_to='none',
        dataloader_num_workers=2,dataloader_pin_memory=True,dataloader_persistent_workers=False,
        remove_unused_columns=False,label_names=['labels'],seed=config['seed'],data_seed=config['seed'],
        optim='adamw_torch',save_safetensors=True)
    return Seq2SeqTrainer(model=model,args=args,train_dataset=Clips(train,root,True),
        eval_dataset=Clips(dev,root,False),data_collator=Collator(proc),processing_class=proc,
        compute_metrics=metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

def smoke_gradients(trainer):
    import torch
    batch=trainer.data_collator([trainer.train_dataset[0]])
    batch={k:v.to(trainer.args.device) for k,v in batch.items()}
    trainer.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    trainer.model.train();trainer.model.zero_grad(set_to_none=True)
    with trainer.compute_loss_context_manager():loss=trainer.model(**batch).loss
    if not torch.isfinite(loss): raise RuntimeError('Nonfinite smoke loss')
    loss.backward()
    grads=[p.grad for n,p in trainer.model.named_parameters() if p.requires_grad and 'lora_' in n]
    if not grads or any(g is None or not torch.isfinite(g).all() for g in grads):
        raise RuntimeError('Missing/nonfinite LoRA gradient')
    if not any(torch.count_nonzero(g)>0 for g in grads): raise RuntimeError('All LoRA gradients zero')
    trainer.model.zero_grad(set_to_none=True)
    return float(loss.detach())

In [ ]:
%%writefile rme_bench.py
"""Reference-blind ASR inference, fingerprinted resume, explicit coverage and timing."""
import gc,time
from pathlib import Path
import numpy as np
from rme_core import BASE,SR,digest,file_hash,write_json
from rme_train import load_wave

def model_fingerprint(path):
    p=Path(path)
    files=sorted(f for f in p.rglob('*') if f.is_file() and f.suffix in ('.json','.safetensors','.bin'))
    if not any(f.suffix in ('.safetensors','.bin') for f in files): raise ValueError('No model weights: '+str(p))
    return digest([(str(f.relative_to(p)),file_hash(f)) for f in files])

def infer(rows,root,target,revision,out,decode,batch_size=2):
    import torch
    from transformers import WhisperProcessor,WhisperForConditionalGeneration
    from peft import PeftModel
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    is_base=target==BASE
    adapter=not is_base and (Path(target)/'adapter_config.json').exists()
    fingerprint=revision if is_base else model_fingerprint(target)
    run_id=digest({'model':fingerprint,'revision':revision,'decode':decode,
                   'data':[(r['id'],r['audio_sha256']) for r in rows],'batch_size':batch_size,
                   'engine':'transformers-4.57.3-fp16-sdpa','torch':torch.__version__,
                   'code':[file_hash(Path(__file__).parent/x) for x in ('rme_core.py','rme_train.py','rme_bench.py')],
                   'gpu':torch.cuda.get_device_name(0)})
    cache=out/(run_id+'.json')
    data=__import__('json').loads(cache.read_text()) if cache.exists() else {'run_id':run_id,'records':{},'complete':False}
    if data['complete']:
        if set(data['records'])!={r['id'] for r in rows}: raise ValueError('Corrupt cache coverage')
        return data
    proc=WhisperProcessor.from_pretrained(BASE if is_base or adapter else target,**({'revision':revision} if is_base or adapter else {}))
    model=WhisperForConditionalGeneration.from_pretrained(BASE if is_base or adapter else target,
        **({'revision':revision} if is_base or adapter else {}),torch_dtype=torch.float16,attn_implementation='sdpa')
    if adapter:model=PeftModel.from_pretrained(model,target).merge_and_unload()
    model=model.to('cuda').eval();model.config.use_cache=True;model.config.apply_spec_augment=False
    model.generation_config.forced_decoder_ids=None
    def predict(waves):
        batch=proc.feature_extractor(waves,sampling_rate=SR,return_tensors='pt',return_attention_mask=True)
        batch={k:v.to('cuda',dtype=torch.float16 if k=='input_features' else v.dtype) for k,v in batch.items()}
        with torch.inference_mode():tokens=model.generate(**batch,**decode)
        return proc.batch_decode(tokens,skip_special_tokens=True),tokens
    pending=[r for r in rows if r['id'] not in data['records']]
    if pending:predict([load_wave(Path(root)/pending[0]['audio_file'])])
    torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
    for start in range(0,len(pending),batch_size):
        part=pending[start:start+batch_size]
        torch.cuda.synchronize();t0=time.perf_counter()
        waves=[load_wave(Path(root)/r['audio_file']) for r in part]
        texts,tokens=predict(waves)
        torch.cuda.synchronize();elapsed=time.perf_counter()-t0
        for r,text,seq in zip(part,texts,tokens):
            # Timing is batch wall time, never advertised as single-request latency.
            capped=len(seq)>=decode['max_length'] and int(seq[-1]) not in (proc.tokenizer.eos_token_id,proc.tokenizer.pad_token_id)
            data['records'][r['id']]={'hyp':text,'seconds':r['seconds'],'batch_s':elapsed,
                'batch_n':len(part),'token_cap_hit':capped,'audio_sha256':r['audio_sha256']}
        write_json(cache,data)
    # Batch=1 latency sample, deterministic and separate from throughput. Includes decode/preprocessing.
    lat=[];rtfs=[]
    sample=sorted(rows,key=lambda r:digest(r['id']))[:min(100,len(rows))]
    for r in sample:
        torch.cuda.synchronize();t0=time.perf_counter()
        predict([load_wave(Path(root)/r['audio_file'])])
        torch.cuda.synchronize();dt=time.perf_counter()-t0
        lat.append(dt);rtfs.append(dt/r['seconds'])
    wall=sum(v['batch_s']/v['batch_n'] for v in data['records'].values())
    seconds=sum(r['seconds'] for r in rows)
    data.update(complete=True,timing={'warm_model_batch1_n':len(lat),'latency_p50_s':float(np.quantile(lat,.5)),
        'latency_p95_s':float(np.quantile(lat,.95)),'batch1_rtf_p95':float(np.quantile(rtfs,.95)),
        'batch_size':batch_size,'throughput_rtf':wall/seconds,'peak_allocated_gb':torch.cuda.max_memory_allocated()/2**30,
        'note':'Local audio decode + feature extraction + generation; excludes model load, queue, network, RME editing.'})
    write_json(cache,data)
    del model,proc;gc.collect();torch.cuda.empty_cache()
    return data

def challenge_infer(manifest, root, target, revision, out):
    """Optional independent real silence/noise/long-form cohort. HF overlap-chunked ASR, no raw truncation."""
    import json,math,soundfile as sf,torch
    from scipy.signal import resample_poly
    from transformers import WhisperProcessor,WhisperForConditionalGeneration,pipeline
    rows=json.loads(Path(manifest).read_text())
    if not rows or len({r['id'] for r in rows})!=len(rows):raise ValueError('Empty/duplicate challenge IDs')
    # Caller supplies independent approved cohort; this function cannot prove independence.
    proc=WhisperProcessor.from_pretrained(target)
    model=WhisperForConditionalGeneration.from_pretrained(target,torch_dtype=torch.float16,attn_implementation='sdpa')
    model.config.use_cache=True;model.config.apply_spec_augment=False;model.generation_config.forced_decoder_ids=None
    pipe=pipeline('automatic-speech-recognition',model=model,tokenizer=proc.tokenizer,
                  feature_extractor=proc.feature_extractor,device=0)
    results=[]
    for r in rows:
        if r['kind'] not in ('silence','noise','long_form','clinical'):raise ValueError('Unknown challenge kind')
        p=(Path(root)/r['audio_file']).resolve()
        if not p.is_relative_to(Path(root).resolve()):raise ValueError('Audio outside challenge root')
        x,sr=sf.read(p,dtype='float32',always_2d=True);x=x.mean(1)
        if not len(x) or not np.isfinite(x).all():raise ValueError('Invalid challenge audio')
        dur=len(x)/sr
        if r['kind'] in ('silence','noise') and r['ref'].strip():raise ValueError('Nonspeech reference must be empty')
        if r['kind']=='long_form' and dur<=30:raise ValueError('long_form must exceed 30 s')
        if sr!=SR:
            g=math.gcd(sr,SR);x=resample_poly(x,SR//g,sr//g)
        torch.cuda.synchronize();t=time.perf_counter()
        h=pipe({'raw':x,'sampling_rate':SR},chunk_length_s=30,stride_length_s=5,return_timestamps=True,
               generate_kwargs={'language':'indonesian','task':'transcribe','num_beams':1,'do_sample':False,'max_length':448})
        torch.cuda.synchronize()
        results.append(dict(r,hyp=h['text'],seconds=dur,wall_s=time.perf_counter()-t,audio_sha256=file_hash(p)))
    write_json(out,{'records':results,'model':model_fingerprint(target),'base_revision':revision,
                    'protocol':'HF overlapping chunks 30s/5s; independent diagnostic; no streaming claim'})
    del pipe,model,proc;gc.collect();torch.cuda.empty_cache()
    return results

In [ ]:
%%writefile test_rme_core.py
import io,tarfile
import pytest
from rme_core import *

def test_numbers_and_negation_survive():
    assert normalize('Tidak, dosis 0,5 mg; TD 120/80.')=='tidak dosis 0.5 mg td 120/80'
    assert score('0,5 mg','5 mg')['err']==1
    assert score('tidak demam','demam')['d']==1

def test_empty_reference_insertions():
    m=corpus_metrics([{'id':'a','ref':''},{'id':'b','ref':'baik'}],{'a':'obat','b':'baik'})
    assert m['wer']==100 and m['i']==1

def test_no_silent_prediction_omission():
    with pytest.raises(ValueError): corpus_metrics([{'id':'a','ref':'obat'}],{})
    with pytest.raises(ValueError): corpus_metrics([{'id':'a','ref':'obat'}],{'a':None})

def test_cluster_pairing():
    rows=[{'id':str(i),'ref':'tidak demam','cluster_id':str(i//2)} for i in range(10)]
    a={r['id']:'tidak demam' for r in rows};b={r['id']:'demam' for r in rows}
    result=paired_cluster_ci(rows,a,b,B=100)
    assert result['clusters']==5 and result['ci95']==[-50.,-50.]
    with pytest.raises(ValueError): paired_cluster_ci(rows,a,{'x':'demam'})

def test_group_connected_components_and_synthetic_train_only():
    rows=[]
    for i in range(100):
        rows.append(dict(id=str(i),group_id='video'+str(i),speaker_id='speaker'+str(i//2),
             synthetic=False,pcm_sha256=str(i),sr=16000,ref='kata '+str(i)))
    rows.append(dict(id='syn',group_id='syn',speaker_id='',synthetic=True,pcm_sha256='syn',sr=16000,ref='sintetis'))
    out,dropped=grouped_split(rows)
    for key in ('group_id','speaker_id','pcm_sha256'):
        m={}
        for r in out:
            if r.get(key):
                assert m.setdefault(r[key],r['split'])==r['split']
    assert next(r for r in out if r['id']=='syn')['split']=='train'
    assert grouped_split(list(reversed(rows)))[0]==out

def test_dedup_heldout_text():
    rows=[dict(id=str(i),group_id=str(i),synthetic=False,pcm_sha256=str(i),sr=16000,
        ref='kalimat unik '+str(i)) for i in range(100)]
    # Duplicate speech joins same group; only one copy is retained.
    rows[1]['pcm_sha256']=rows[0]['pcm_sha256']
    out,drop=grouped_split(rows)
    assert len(out)==99 and drop[0]['drop_reason']=='duplicate_pcm'

def test_fail_closed_gate():
    assert release_status({})['status']=='NOT_READY'
    assert release_status({'human_gold_references':'True'})['status']=='NOT_READY'

def test_tar_traversal(tmp_path):
    p=tmp_path/'bad.tar'
    with tarfile.open(p,'w') as t:
        m=tarfile.TarInfo('../escape');m.size=1;t.addfile(m,io.BytesIO(b'x'))
    with pytest.raises(ValueError): extract_safe(p,tmp_path/'dest')

def test_youtube_underscore_group(tmp_path):
    audio=tmp_path/'train';audio.mkdir();(audio/'lvl3_chk_001_-qUTxy5a9_E.mp3').write_bytes(b'a')
    manifest=tmp_path/'manifest.json'
    write_json(manifest,[{'audio_file':'lvl3_chk_001_-qUTxy5a9_E.mp3','text':'nyeri'}])
    r=canonical_rows([manifest],tmp_path)[0]
    assert r['video_id']=='-qUTxy5a9_E'

def test_reject_ambiguous_basename(tmp_path):
    for s in ('train','test'):
        p=tmp_path/s;p.mkdir();(p/'same.wav').write_bytes(b'x')
    manifest=tmp_path/'manifest.json';write_json(manifest,[{'audio_file':'same.wav','text':'halo'}])
    with pytest.raises(ValueError):canonical_rows([manifest],tmp_path)

def test_numeric_proxy_not_semantic_certificate():
    assert critical_proxy('dosis 0,5 mg','dosis 5 mg')['number']
    assert critical_proxy('tidak demam','demam')['negation']
    assert not critical_proxy('tidak batuk, demam','batuk, tidak demam')['negation']

def test_term_precision_penalizes_invented_drug():
    m=term_metrics([{'id':'1','ref':'parasetamol'}],{'1':'parasetamol amoksisilin'},['parasetamol','amoksisilin'])
    assert m['recall']==1 and m['precision']==.5
    assert term_metrics([],{},[])['status']=='NOT_MEASURED'

def test_conflicting_same_audio_quarantines_both():
    rows=[{'sr':16000,'pcm_sha256':'same','ref':s,'reject_reasons':[]} for s in ('perut mulas','dada sesak')]
    assert all('identical_audio_conflicting_references' in r['reject_reasons'] for r in quarantine_conflicts(rows))

def test_manifest_uses_original_not_llm_rewrite(tmp_path):
    (tmp_path/'clean_chk_1_abcdefghijk.mp3').touch()
    p=tmp_path/'manifest.json'
    write_json(p,[{'audio_file':'clean_chk_1_abcdefghijk.mp3','raw_text':'tidak demam 0,5 mg',
        'clean_text':'demam 5 mg','group_id':'abcdefghijk','duration':3}])
    r=canonical_rows([p],tmp_path)[0]
    assert r['ref']=='tidak demam 0,5 mg' and r['declared_seconds']==3
    assert 'clean_changed_number' in r['label_review_flags']
    assert 'clean_changed_negation' in r['label_review_flags']

def test_audio_duration_mismatch_quarantined(tmp_path):
    import soundfile as sf
    sf.write(tmp_path/'x.wav',np.ones(16000,dtype=np.float32)*.1,16000)
    r=inspect_audio({'audio_file':'x.wav','ref':'nyeri dada','declared_seconds':2},tmp_path)
    assert 'duration_metadata_mismatch_requires_review' in r['reject_reasons']
    assert r['seconds']==1.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','test_rme_core.py'],check=True)
from rme_core import *

## 4. Dataset yang sama

In [ ]:
import gdown
if not ARCHIVE.exists():
    result=gdown.download(id=FILE_ID,output=str(ARCHIVE),quiet=False,resume=True)
    if result is None: raise RuntimeError('Unduhan gagal; salin arsip ke ARCHIVE dari Drive')
archive_sha=file_hash(ARCHIVE)
if archive_sha!=EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError('Arsip berbeda/tidak lengkap. Periksa unduhan; audit ulang jika dataset sengaja berubah.')
marker=ROOT/'.archive_sha256'
if not marker.exists():
    extract_safe(ARCHIVE,ROOT)
    marker.write_text(archive_sha)
elif marker.read_text()!=archive_sha: raise RuntimeError('Gunakan ROOT baru untuk arsip berbeda')
# Manifest ditetapkan berdasarkan audit arsip, bukan rilis GitHub lama.
MANIFEST_NAMES = ['manifest_rme_train_gold.json', 'manifest_rme_dev_gold.json', 'manifest_rme_test_gold.json']
manifests=[]
for name in MANIFEST_NAMES:
    candidates=list(ROOT.rglob(name))
    if len(candidates)!=1:raise RuntimeError('Manifest tidak ditemukan/ambigu: '+name)
    manifests.extend(candidates)
rows=canonical_rows(manifests,ROOT)
print('Baris manifest:',len(rows),'| sumber:',dict(collections.Counter(r['source'] for r in rows)))

## 5. Verifikasi provenance dan rencana uji

In [ ]:
PLAN_PATH=WORK/'split_plan.json';MODEL_INFO=WORK/'latest_model.json'
if not PLAN_PATH.exists() or not MODEL_INFO.exists():raise RuntimeError('Jalankan training v3 dahulu')
plan=json.loads(PLAN_PATH.read_text());info=json.loads(MODEL_INFO.read_text())
check=dict(plan);check.pop('fingerprint')
if digest(check)!=plan['fingerprint']:raise RuntimeError('Split plan rusak/diubah')
if info['plan_fingerprint']!=plan['fingerprint'] or plan['archive_sha256']!=archive_sha:
    raise RuntimeError('Model/dataset/split tidak cocok')
from rme_bench import infer,model_fingerprint
if model_fingerprint(info['model_dir'])!=info['model_sha256']:raise RuntimeError('Bobot model telah berubah')
revision=info['revision'];DECODE=info['decode']
test_rows=[r for r in plan['rows'] if r['split']=='test']
assert test_rows and not any(r['synthetic'] for r in test_rows)
# Verify actual file content against locked plan before any cached predictions can be reused.
for r in test_rows:
    if file_hash(ROOT/r['audio_file'])!=r['audio_sha256']:raise RuntimeError('Audio berubah: '+r['id'])
OUT=WORK/'benchmarks'/info['run_id'];OUT.mkdir(parents=True,exist_ok=True)
print('Planned:',len(test_rows),'clips;',len({r['cluster_id'] for r in test_rows}),'groups')
print('Excluded before split:',{k:plan[k] for k in ['rejected_audio','rejected_tokens','dedup_removed']})

## 6. Inferensi mentah, tanpa reference prompt atau koreksi otomatis
Satu model dimuat pada satu waktu. Cache terikat hash bobot/audio, revisi, GPU, dan decoding.
Kegagalan audio/inference menghentikan run; tidak diam-diam membuang klip sulit dari skor.
Waktu batch untuk throughput dipisahkan dari sampel latency batch=1; keduanya sudah warm-up.

In [ ]:
RUNS={}
for name,target in [('base',BASE),('rme_v3',info['model_dir'])]:
    RUNS[name]=infer(test_rows,ROOT,target,revision,OUT/name,DECODE,batch_size=2)
HYPS={name:{k:v['hyp'] for k,v in run['records'].items()} for name,run in RUNS.items()}

## 7. WER/CER, S/D/I, CI kelompok, dan irisan sumber

In [ ]:
metrics={name:corpus_metrics(test_rows,h) for name,h in HYPS.items()}
refs={r['id']:r['ref'] for r in test_rows}
for name,h in HYPS.items():
    metrics[name]['wer_cluster_ci95']=paired_cluster_ci(test_rows,h,refs)['ci95']
    metrics[name]['timing']=RUNS[name]['timing']
    metrics[name]['token_cap_hits']=sum(v['token_cap_hit'] for v in RUNS[name]['records'].values())
paired=paired_cluster_ci(test_rows,HYPS['rme_v3'],HYPS['base'])
slices={}
for source in sorted({r['source'] for r in test_rows}):
    rr=[r for r in test_rows if r['source']==source]
    slices[source]={name:corpus_metrics(rr,{r['id']:h[r['id']] for r in rr}) for name,h in HYPS.items()}
print(json.dumps({'metrics':metrics,'paired_v3_minus_base':paired,'sources':slices},indent=2))
print('CI delta < 0 mendukung perbaikan pada korpus ini; bukan sertifikasi klinis. Tidak menghitung p-value bootstrap semu.')

## 8. Review buta dan flag klinis
`blind_reference_review.csv` tidak memperlihatkan hipotesis model kepada pembuat referensi.
Dokter mendengar audio dan mengisi transkrip verbatim; reviewer kedua mengadjudikasi.
Simpan versi final, identitas reviewer pseudonim, dan status verifikasi. Jangan mengisi referensi dari output model.
`clinical_error_review.csv` digunakan sesudah referensi dikunci untuk menilai obat, dosis, unit, negasi,
alergi, subjek/temporality, dan dampak klinis. Flag regex hanya screening, bukan akurasi semantik.

In [ ]:
import csv
with (OUT/'blind_reference_review.csv').open('w',newline='') as f:
    w=csv.writer(f);w.writerow(['id','audio_file','source','reference_verbatim','reviewer_1','reviewer_2','adjudicated'])
    for r in test_rows:w.writerow([r['id'],r['audio_file'],r['source'],'','','',''])
flags=[]
with (OUT/'clinical_error_review.csv').open('w',newline='') as f:
    w=csv.writer(f);w.writerow(['id','source','reference_dataset','hyp_raw','number_flag','unit_flag','negation_flag',
                              'drug_error','dose_error','allergy_error','scope_error','severity','adjudicated'])
    for r in test_rows:
        h=HYPS['rme_v3'][r['id']];p=critical_proxy(r['ref'],h);flags.append(p)
        w.writerow([r['id'],r['source'],r['ref'],h,p['number'],p['unit'],p['negation'],'','','','','',''])
proxy={k:sum(p[k] for p in flags) for k in ('number','unit','negation')}
print('Screening discrepancy counts:',proxy)
# Optional doctor-corrected references; exact ID coverage + two reviewers required.
VERIFIED_CSV=OUT/'blind_reference_review_filled.csv'
verified_metrics=None
if VERIFIED_CSV.exists():
    vv=list(csv.DictReader(VERIFIED_CSV.open()))
    if len({r['id'] for r in vv})!=len(vv) or {r['id'] for r in vv}!={r['id'] for r in test_rows}:
        raise RuntimeError('Verified reference coverage mismatch')
    for v in vv:
        if v['adjudicated'].lower()!='true' or not v['reviewer_1'] or not v['reviewer_2'] or v['reviewer_1']==v['reviewer_2']:
            raise RuntimeError('Review/adjudication belum lengkap')
    mapping={r['id']:r['reference_verbatim'] for r in vv}
    gold=[dict(r,ref=mapping[r['id']]) for r in test_rows]
    verified_metrics={name:corpus_metrics(gold,h) for name,h in HYPS.items()}
    verified_metrics['reference_file_sha256']=file_hash(VERIFIED_CSV)
    print('Doctor reference metrics:',verified_metrics)

## 9a. Opsional: kamus klinis dan cohort challenge independen
Letakkan `clinical_terms.json` di WORK berupa list istilah yang telah diperiksa klinisi (nama obat,
diagnosis, prosedur). Tidak memakai daftar istilah dari test untuk mengubah prompt model.
Letakkan `challenge_manifest.json` di WORK, audio di `WORK/challenge_audio/`.
Format setiap record: id, audio_file, ref, kind (`silence`, `noise`, `long_form`, `clinical`).
Referensi silence/noise harus kosong dan sudah dipastikan tidak ada ucapan; long_form harus >30 detik.
Gunakan rekaman nyata yang belum pernah dilatih, bukan hanya gelombang nol sintetis.
Inferensi long-form memakai overlap 30 s/stride 5 s. Ini uji offline, bukan bukti streaming.

In [ ]:
TERMS_PATH=WORK/'clinical_terms.json'
terms=json.loads(TERMS_PATH.read_text()) if TERMS_PATH.exists() else []
term_scores={name:term_metrics(test_rows,h,terms) for name,h in HYPS.items()}
write_json(OUT/'term_metrics.json',term_scores)
CHALLENGE=WORK/'challenge_manifest.json'
if CHALLENGE.exists():
    from rme_bench import challenge_infer
    cr=challenge_infer(CHALLENGE,WORK/'challenge_audio',info['model_dir'],revision,OUT/'challenge_predictions.json')
    ch={}
    for kind in sorted({r['kind'] for r in cr}):
        rr=[r for r in cr if r['kind']==kind]
        if kind in ('silence','noise'):
            k=sum(bool(normalize(r['hyp'])) for r in rr)
            ch[kind]={'n':len(rr),'nonempty_outputs':k,'false_activation_rate':k/len(rr),
                      'inserted_words':sum(len(normalize(r['hyp']).split()) for r in rr)}
        else:ch[kind]=corpus_metrics(rr,{r['id']:r['hyp'] for r in rr})
    write_json(OUT/'challenge_metrics.json',ch)
    print(ch)
else:print('Challenge belum tersedia: silence/noise dan long-form belum tervalidasi.')

## 9. Bukti yang belum tersedia dan laporan
Benchmark ini **tidak otomatis lulus enterprise**. Dataset pendidikan/sintetis bukan pengganti uji dikte
RS/klinik dari dokter, perangkat, ruangan dan lokasi yang tidak terlihat saat training.
Sebelum pilot: tetapkan ambang WER, error kritis dan SLA latency bersama pemilik klinis, lalu uji
rekaman nyata termasuk noise/silence, aksen, nama obat, dosis/desimal, koreksi diri, alergi, negasi, serta sesi >30 detik.
Zero error pada sampel kecil tidak membuktikan zero risk. Untuk 0 error dari n kesempatan independen,
batas atas satu sisi 95% kira-kira 3/n (mis. n=3000 → ~0,1%); peluang dalam satu sesi saling berkorelasi.

In [ ]:
# Evidence left unavailable deliberately; a checkbox cannot create clinical evidence.
evidence={'human_gold_references':verified_metrics is not None}
verdict=release_status(evidence)
report={'model':info,'test_clips':len(test_rows),'metrics':metrics,'paired':paired,'slices':slices,
        'critical_screening_counts':proxy,'verified_metrics':verified_metrics,'release':verdict,
        'limits':['unknown speakers may recur across different videos','no independent hospital dictation cohort',
                  'no real nonspeech challenge set','no long-form/streaming validation','no clinician critical-error adjudication yet']}
write_json(OUT/'benchmark_report.json',report)
print(json.dumps(verdict,indent=2));print('Report:',OUT/'benchmark_report.json')